"""
Build the HMM estimation panel with sugar-based yogurt variables.

Input : hmm_estimation_panel.parquet            (fat-based yogurt variables)
        yogurt_sugar_trip_level_2008_2017.tsv   (sugar-based trip file from script 04)
Output: hmm_estimation_panel_sugar.parquet      (same folder as the input panel)

Join key: (hhid, trip_id, panel_year, purchase_date)

Replacement rule:
    yogurt_y <- yogurt_sugar_y
        0 = no yogurt on the trip
        1 = trip contains at least one regular-sugar yogurt
        2 = trip contains yogurt, all high-sugar
    Dropped: skimmed_yogurt_spending, low_fat_yogurt_spending, other_yogurt_spending
    Added  : high_sugar_spending, regular_sugar_spending

Trips with no row in the sugar file get yogurt_y = 0 and zero spending.
The script stops before writing if the panel has a yogurt trip that the
sugar file does not cover, because that would signal a bug upstream.

Requires: pip install duckdb
"""

In [1]:


from pathlib import Path
import duckdb

PANEL_PATH = Path(r"C:\Users\rwen13\Documents\healthy_purchase_data\processed\hmm_input\hmm_estimation_panel.parquet")
SUGAR_TRIP_PATH = Path(r"C:\Users\rwen13\Documents\healthy_purchase_data\processed\yogurt_sugar_trip_level\yogurt_sugar_trip_level_2008_2017.tsv")
OUT_PATH = PANEL_PATH.with_name("hmm_estimation_panel_sugar.parquet")

TAB = "\t"
KEYS = "(hhid, trip_id, panel_year, purchase_date)"
MATCH = (
    "p.hhid = s.hhid AND p.trip_id = s.trip_id "
    "AND p.panel_year = s.panel_year AND p.purchase_date = s.purchase_date"
)


def as_sql_path(path: Path) -> str:
    # DuckDB accepts forward slashes on Windows, which avoids escaping backslashes
    return path.as_posix()


con = duckdb.connect()

# 1. Sugar trip file.
# Read everything as text first, then cast. This keeps hhid and trip_id
# as strings so leading zeros are preserved.
con.execute(f"""
    CREATE TABLE sugar AS
    SELECT
        trim(hhid)                             AS hhid,
        CAST(panel_year AS INTEGER)            AS panel_year,
        trim(trip_id)                          AS trip_id,
        CAST(purchase_date AS DATE)            AS purchase_date,
        CAST(yogurt_sugar_y AS TINYINT)        AS yogurt_sugar_y,
        CAST(high_sugar_spending AS DOUBLE)    AS high_sugar_spending,
        CAST(regular_sugar_spending AS DOUBLE) AS regular_sugar_spending
    FROM read_csv('{as_sql_path(SUGAR_TRIP_PATH)}',
                  delim='{TAB}', header=true, all_varchar=true)
""")

# The join key must be unique in the sugar file, otherwise the join would duplicate panel rows.
dup_keys = con.execute("""
    SELECT count(*) - (SELECT count(*) FROM (SELECT DISTINCT hhid, trip_id, panel_year, purchase_date FROM sugar))
    FROM sugar
""").fetchone()[0]
if dup_keys:
    raise RuntimeError(f"Sugar trip file has {dup_keys} duplicate trip keys.")

# 2. Panel as a lazy view. The data is not loaded into memory here.
con.execute(f"CREATE VIEW panel AS SELECT * FROM read_parquet('{as_sql_path(PANEL_PATH)}')")

# 3. Audit before writing.
n_panel = con.execute("SELECT count(*) FROM panel").fetchone()[0]
n_sugar = con.execute("SELECT count(*) FROM sugar").fetchone()[0]
n_matched = con.execute(f"SELECT count(*) FROM panel p JOIN sugar s USING {KEYS}").fetchone()[0]

sugar_not_in_panel = con.execute(f"""
    SELECT count(*) FROM sugar s
    WHERE NOT EXISTS (SELECT 1 FROM panel p WHERE {MATCH})
""").fetchone()[0]

# Panel yogurt trips with no sugar row. This should be zero; if not, stop.
panel_yogurt_unmatched = con.execute(f"""
    SELECT count(*) FROM panel p
    WHERE p.yogurt_y > 0 AND NOT EXISTS (SELECT 1 FROM sugar s WHERE {MATCH})
""").fetchone()[0]

# Panel says "no yogurt" but the sugar file has a yogurt trip. Informational only.
panel_zero_but_sugar = con.execute(f"""
    SELECT count(*) FROM panel p
    JOIN sugar s USING {KEYS}
    WHERE p.yogurt_y = 0
""").fetchone()[0]

print(f"Panel rows                          : {n_panel:,}")
print(f"Sugar trip rows                     : {n_sugar:,}")
print(f"Matched trips                       : {n_matched:,}")
print(f"Sugar trips not in panel            : {sugar_not_in_panel:,}")
print(f"Panel yogurt trips missing in sugar : {panel_yogurt_unmatched:,}")
print(f"Panel yogurt_y=0 but sugar has yogurt: {panel_zero_but_sugar:,}")

if panel_yogurt_unmatched:
    raise RuntimeError("Panel has yogurt trips that the sugar file does not cover. Fix upstream before writing.")

print("\n--- Yearly audit ---")
print(con.execute(f"""
    SELECT
        p.panel_year,
        count(*) FILTER (WHERE p.yogurt_y > 0)   AS panel_yogurt_trips,
        count(s.hhid)                            AS matched_sugar_trips,
        round(sum(s.high_sugar_spending), 2)     AS high_sugar_sum,
        round(sum(s.regular_sugar_spending), 2)  AS regular_sugar_sum
    FROM panel p
    LEFT JOIN sugar s USING {KEYS}
    GROUP BY 1
    ORDER BY 1
""").df().to_string(index=False))

# 4. Replace the fat yogurt variables and write the new panel.
con.execute(f"""
    COPY (
        SELECT
            p.* EXCLUDE (yogurt_y, skimmed_yogurt_spending,
                         low_fat_yogurt_spending, other_yogurt_spending),
            COALESCE(s.yogurt_sugar_y, 0)::TINYINT       AS yogurt_y,
            COALESCE(s.high_sugar_spending, 0.0)         AS high_sugar_spending,
            COALESCE(s.regular_sugar_spending, 0.0)      AS regular_sugar_spending
        FROM panel p
        LEFT JOIN sugar s USING {KEYS}
    ) TO '{as_sql_path(OUT_PATH)}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

# 5. Verify the output.
n_out = con.execute(f"SELECT count(*) FROM read_parquet('{as_sql_path(OUT_PATH)}')").fetchone()[0]
if n_out != n_panel:
    raise RuntimeError(f"Row count changed: panel {n_panel:,} vs output {n_out:,}")

print(f"\nWritten: {OUT_PATH}")
print(f"Rows: {n_out:,} (matches panel)")

Panel rows                          : 37,606,554
Sugar trip rows                     : 6,049,411
Matched trips                       : 4,549,914
Sugar trips not in panel            : 1,499,497
Panel yogurt trips missing in sugar : 0
Panel yogurt_y=0 but sugar has yogurt: 0

--- Yearly audit ---
 panel_year  panel_yogurt_trips  matched_sugar_trips  high_sugar_sum  regular_sugar_sum
       2008              367104               367104       760030.04          629319.65
       2009              418912               418912       836559.04          753594.37
       2010              445897               445897       924506.49          792605.01
       2011              482470               482470      1121605.94          872635.96
       2012              470921               470921      1276545.32          812359.50
       2013              483030               483030      1415176.01          837059.51
       2014              496053               496053      1444206.59          926020.60


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Written: C:\Users\rwen13\Documents\healthy_purchase_data\processed\hmm_input\hmm_estimation_panel_sugar.parquet
Rows: 37,606,554 (matches panel)
